In [1]:
import os
import json
import time
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

In [2]:
load_dotenv(dotenv_path="../.env")
API_KEY = os.getenv("FMP_API_KEY")
BASE = "https://financialmodelingprep.com/stable"

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

tickers = pd.read_csv("../data/tickers.csv")
print("Companies:", len(tickers))
print(tickers["sector"].value_counts())

Companies: 56
sector
Tech              14
Financials        12
Consumer          10
Transport          9
Healthcare         8
Semiconductors     3
Name: count, dtype: int64


In [3]:
ENDPOINTS = [
    "income-statement",
    "balance-sheet-statement",
    "cash-flow-statement",
    "employee-count",
]

def fetch_and_cache(ticker, endpoint, limit=5):
    path = RAW_DIR / f"{ticker}_{endpoint}.json"

    if path.exists():
        return "cached"

    params = {"symbol": ticker, "apikey": API_KEY}
    if endpoint != "employee-count":
        params["limit"] = limit

    response = requests.get(f"{BASE}/{endpoint}", params=params)
    response.raise_for_status()
    data = response.json()

    if not data:
        return "empty"

    with open(path, "w") as f:
        json.dump(data, f)
    return "fetched"

In [4]:
results = []

for row in tickers.itertuples():
    for endpoint in ENDPOINTS:
        try:
            status = fetch_and_cache(row.ticker, endpoint)
        except Exception as e:
            status = f"failed: {type(e).__name__}"
        results.append({"ticker": row.ticker, "endpoint": endpoint, "status": status})
        if status == "fetched":
            time.sleep(0.3)

log = pd.DataFrame(results)
print(log["status"].value_counts())

status
cached    224
Name: count, dtype: int64


In [5]:
problems = log[~log["status"].isin(["cached", "fetched"])]
print("Problem count:", len(problems))
problems

Problem count: 0


,ticker,endpoint,status


In [6]:
def combining_company_statement(statement_type):
    frames = []

    for row in tickers.itertuples():
        path = RAW_DIR / f"{row.ticker}_{statement_type}.json"

        with open(path, "r") as f:
            frames.append(pd.DataFrame(json.load(f)))

    return pd.concat(frames, ignore_index=True)

In [7]:
income = combining_company_statement("income-statement")
balance = combining_company_statement("balance-sheet-statement")
cashflow = combining_company_statement("cash-flow-statement")

for name, df in [("income", income), ("balance", balance), ("cashflow", cashflow)]:
    print(name, df.shape)


income (280, 39)
balance (280, 61)
cashflow (280, 47)


In [8]:
income = income[["symbol", "fiscalYear", "reportedCurrency", "period", "revenue", "grossProfit", 
"operatingIncome", "netIncome", "researchAndDevelopmentExpenses", "netInterestIncome"]]
balance = balance[["symbol", "fiscalYear", "totalDebt", "totalStockholdersEquity"]]
cashflow = cashflow[["symbol", "fiscalYear", "capitalExpenditure", "freeCashFlow"]]

for name, df in [("income", income), ("balance", balance), ("cashflow", cashflow)]:
    print(name, df.shape)

income (280, 10)
balance (280, 4)
cashflow (280, 4)


In [9]:
merged_df = pd.merge(income, balance, on=["symbol", "fiscalYear"], 
how="outer", validate="one_to_one")

merged_df = pd.merge(merged_df, cashflow, on=["symbol", "fiscalYear"], 
how="outer", validate="one_to_one")

print(merged_df.shape)

(280, 14)


In [10]:
employee_count = combining_company_statement("employee-count")

employee_count["year"] = pd.to_datetime(employee_count["periodOfReport"]).dt.year

employee_count = employee_count[employee_count["year"] >= 2020]

employee_count = employee_count.sort_values("filingDate")
employee_count = employee_count.drop_duplicates(subset=["symbol", "year"], keep="last")

employee_count = employee_count[["symbol", "year", "employeeCount"]]

merged_df["fiscalYear"] = merged_df["fiscalYear"].astype(int)

with_employees = pd.merge(
    merged_df,
    employee_count,
    left_on=["symbol", "fiscalYear"],
    right_on=["symbol", "year"],
    how="left",
    validate="one_to_one",
)

with_employees = with_employees.drop(columns=["year"])

print(with_employees.shape)
print("Missing employee counts:", with_employees["employeeCount"].isna().sum())
print(with_employees[with_employees["employeeCount"].isna()])

(280, 15)
Missing employee counts: 1
    symbol  fiscalYear reportedCurrency period       revenue  grossProfit  \
252    UNH        2023              USD     FY  371622000000  90958000000   

     operatingIncome    netIncome  researchAndDevelopmentExpenses  \
252      32358000000  22381000000                               0   

     netInterestIncome    totalDebt  totalStockholdersEquity  \
252        -3246000000  67435000000              88756000000   

     capitalExpenditure  freeCashFlow  employeeCount  
252         -3386000000   25682000000            NaN  


In [11]:
complete_df = pd.merge(
    with_employees,
    tickers,
    left_on="symbol",
    right_on="ticker",
    how="left",
    validate="many_to_one",
)

complete_df = complete_df.drop(columns=["ticker"])

print(complete_df["reportedCurrency"].value_counts())
print(complete_df["period"].value_counts())

print(complete_df.shape)
complete_df.columns


reportedCurrency
USD    280
Name: count, dtype: int64
period
FY    280
Name: count, dtype: int64
(280, 17)


Index(['symbol', 'fiscalYear', 'reportedCurrency', 'period', 'revenue',
       'grossProfit', 'operatingIncome', 'netIncome',
       'researchAndDevelopmentExpenses', 'netInterestIncome', 'totalDebt',
       'totalStockholdersEquity', 'capitalExpenditure', 'freeCashFlow',
       'employeeCount', 'name', 'sector'],
      dtype='object')

In [12]:
complete_df.to_csv("../data/financials.csv", index=False)